[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/validation/VIX_FEATURE_SELECTION.ipynb)


# VIX Feature Selection Methods v1 — SHAP vs RFE vs LASSO, sur la config de référence

**Rôle.** Depuis le début du projet, la sélection de features est faite exclusivement via
SHAP (top-N par importance moyenne |valeur SHAP|). Ce notebook teste si **la méthode de
sélection elle-même** est un facteur limitant, en la comparant à deux alternatives
classiques du cours de Machine Learning (Nédra Mellouli) : RFE (wrapper, élimination
récursive) et LASSO (embedded, pénalité L1) — à isométrie stricte de tout le reste
(même config, même classifieur, même sampler, mêmes folds walk-forward).

**Config de référence fixée** (celle du README) : régime GLOBAL, horizon 5 jours,
N=8 features, RandomForest, SMOTE — F1_dir≈0.610±0.025 avec SHAP.

**Méthodes comparées** :
- **SHAP** (actuelle) : importance moyenne |valeur SHAP| d'un XGBoost pilote entraîné
  sur le pool pré-filtré.
- **RFE** (wrapper) : élimination récursive de features avec une régression logistique
  multinomiale comme estimateur de base (`sklearn.feature_selection.RFE`).
- **LASSO** (embedded) : régression logistique multinomiale pénalisée L1, sélection des
  N plus grands |coefficient| moyennés sur les classes.

Les trois méthodes partent du **même pré-filtre** (top 450 par importance XGBoost, comme
le fait déjà `shap_rank` dans le reste du projet) pour isoler l'effet de l'étape finale
de sélection, pas du pré-filtre commun.


In [1]:
import subprocess, sys
pkgs = ['xgboost', 'shap', 'xlsxwriter', 'imbalanced-learn', 'pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


Installation OK


In [2]:
import os, time, json, warnings, random, subprocess
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_FEATURE_SELECTION'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'horizon': 5,             # config de référence établie
    'regime': 'GLOBAL',
    'N': 8,                    # config de référence établie
    'algo': 'RandomForest',
    'sampler': 'SMOTE',
    'methods': ['SHAP', 'RFE', 'LASSO'],
    'n_wf_folds': 5,
    'min_train_frac': 0.40,   # doit matcher VIX_FINAL_FEATURES
    'shap_sample': 500,
    'pool_prefilter': 450,
    'rfe_step': 0.1,
    'lasso_C': 0.5,
    'min_train_rows': 100, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
RESULTS_CSV = 'vix_feature_selection_results.csv'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-feature-selection'

BASELINE_F1_DIR = 0.610  # référence SHAP (README), pour rappel dans la synthèse

print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | régime={CONFIG['regime']} h={CONFIG['horizon']}j "
      f"N={CONFIG['N']} algo={CONFIG['algo']} sampler={CONFIG['sampler']} | "
      f"méthodes: {CONFIG['methods']}")


VIX_FEATURE_SELECTION v1 | régime=GLOBAL h=5j N=8 algo=RandomForest sampler=SMOTE | méthodes: ['SHAP', 'RFE', 'LASSO']


In [3]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb en premier "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL = meta['feature_pool']
VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool: {len(FEATURE_POOL)} features "
      f"(dont {len(meta['interaction_features'])} interactions) | source: {meta['date_min']} → {meta['date_max']}")

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


[PULL OK] Dataset récupéré depuis 'results/vix-final-features'
Dataset: (6908, 1300) | VIX=IDX_VIX | pool: 1102 features (dont 21 interactions) | source: 2000-01-03 → 2026-07-21
  Fold 1: train → 2010-08-16 | test 2010-08-17 → 2013-10-21
  Fold 2: train → 2013-10-21 | test 2013-10-22 → 2016-12-28
  Fold 3: train → 2016-12-28 | test 2016-12-29 → 2020-03-06
  Fold 4: train → 2020-03-06 | test 2020-03-09 → 2023-05-12
  Fold 5: train → 2023-05-12 | test 2023-05-15 → 2026-07-21


In [4]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

def get_clf():
    return RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                  class_weight='balanced', random_state=SEED, n_jobs=-1)

def prefilter_pool(X_tr, y_tr, prefilter):
    nf = X_tr.shape[1]
    if nf <= prefilter:
        return np.arange(nf)
    pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                       eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pf.fit(X_tr, y_tr)
    return np.argsort(pf.feature_importances_)[::-1][:prefilter]

def select_shap(X_tr, y_tr, n):
    keep = prefilter_pool(X_tr, y_tr, CONFIG['pool_prefilter'])
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    order = np.argsort(np.asarray(arr).ravel())[::-1][:n]
    return keep[order]

def select_rfe(X_tr, y_tr, n):
    keep = prefilter_pool(X_tr, y_tr, CONFIG['pool_prefilter'])
    Xk = X_tr[:, keep]
    est = LogisticRegression(max_iter=200, random_state=SEED)
    sel = RFE(est, n_features_to_select=n, step=CONFIG['rfe_step']).fit(Xk, y_tr)
    return keep[np.where(sel.support_)[0]]

def select_lasso(X_tr, y_tr, n):
    keep = prefilter_pool(X_tr, y_tr, CONFIG['pool_prefilter'])
    Xk = X_tr[:, keep]
    est = LogisticRegression(penalty='l1', solver='saga', C=CONFIG['lasso_C'], max_iter=1000,
                             random_state=SEED, n_jobs=-1)
    est.fit(Xk, y_tr)
    importance = np.abs(est.coef_).mean(axis=0)
    order = np.argsort(importance)[::-1][:n]
    return keep[order]

SELECTORS = {'SHAP': select_shap, 'RFE': select_rfe, 'LASSO': select_lasso}
print("Helpers OK (build_target, metrics, get_clf, prefilter_pool, select_shap/rfe/lasso)")


Helpers OK (build_target, metrics, get_clf, prefilter_pool, select_shap/rfe/lasso)


In [5]:
# ============================================================
# MOTEUR : par fold walk-forward, 3 méthodes de sélection (SHAP/RFE/LASSO) -> même
# classifieur (RandomForest+SMOTE, N=8) -> métriques, à isométrie stricte.
# ============================================================
rows = []
t0 = time.time()

for k in range(CONFIG['n_wf_folds']):
    cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
    cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
    target, reg_r, _ = build_target(df_features[VIX_COL], CONFIG['horizon'], cut)
    idx = target.index
    tr_mask = np.asarray(idx < cut_date)
    te_mask = np.asarray((idx >= cut_date) & (idx <= nxt_date))
    y_tr = target.values[tr_mask].astype(int); y_te = target.values[te_mask].astype(int)
    if len(y_tr) < CONFIG['min_train_rows'] or len(y_te) < CONFIG['min_test_rows']:
        print(f"Fold {k+1}: pas assez de données — ignoré.")
        continue

    X_pool = df_features[FEATURE_POOL].reindex(idx)
    sc = RobustScaler()
    X_tr = sc.fit_transform(np.nan_to_num(X_pool.values[tr_mask]))
    X_te = sc.transform(np.nan_to_num(X_pool.values[te_mask]))

    print(f"\nFold {k+1}: train={len(y_tr)} test={len(y_te)}")
    for method in CONFIG['methods']:
        cols = SELECTORS[method](X_tr, y_tr, CONFIG['N'])
        Xtr_n, Xte_n = X_tr[:, cols], X_te[:, cols]
        try:
            Xr, yr = SMOTE(random_state=SEED).fit_resample(Xtr_n, y_tr)
        except Exception:
            Xr, yr = Xtr_n, y_tr
        clf = get_clf(); clf.fit(Xr, yr)
        met = metrics(y_te, clf.predict(Xte_n))
        feat_names = [FEATURE_POOL[c] for c in cols]
        rows.append({'fold': k + 1, 'method': method, 'n_test': len(y_te),
                     'test_start': str(cut_date.date()), 'test_end': str(nxt_date.date()),
                     'features': '|'.join(feat_names), **met})
        print(f"  {method:6s} F1_dir={met['F1_dir']:.3f} F1_UP_FORT={met['F1_UP_FORT']} "
              f"F1_DOWN_FORT={met['F1_DOWN_FORT']}")

df_fsel = pd.DataFrame(rows)
df_fsel.to_csv(RESULTS_CSV, index=False)
print(f"\n[TERMINÉ] {len(df_fsel)} lignes en {(time.time()-t0)/60:.1f}min -> {RESULTS_CSV}")



Fold 1: train=2703 test=807
  SHAP   F1_dir=0.582 F1_UP_FORT=0.3642 F1_DOWN_FORT=0.6116
  RFE    F1_dir=0.573 F1_UP_FORT=0.2109 F1_DOWN_FORT=0.6208
  LASSO  F1_dir=0.563 F1_UP_FORT=0.2697 F1_DOWN_FORT=0.6067

Fold 2: train=3510 test=816
  SHAP   F1_dir=0.633 F1_UP_FORT=0.4342 F1_DOWN_FORT=0.6472
  RFE    F1_dir=0.605 F1_UP_FORT=0.5208 F1_DOWN_FORT=0.6219
  LASSO  F1_dir=0.588 F1_UP_FORT=0.6054 F1_DOWN_FORT=0.5779

Fold 3: train=4326 test=818
  SHAP   F1_dir=0.618 F1_UP_FORT=0.3094 F1_DOWN_FORT=0.6552
  RFE    F1_dir=0.564 F1_UP_FORT=0.3162 F1_DOWN_FORT=0.4484
  LASSO  F1_dir=0.624 F1_UP_FORT=0.2286 F1_DOWN_FORT=0.707

Fold 4: train=5144 test=813
  SHAP   F1_dir=0.586 F1_UP_FORT=0.4577 F1_DOWN_FORT=0.545
  RFE    F1_dir=0.546 F1_UP_FORT=0.412 F1_DOWN_FORT=0.4595
  LASSO  F1_dir=0.540 F1_UP_FORT=0.443 F1_DOWN_FORT=0.5488

Fold 5: train=5957 test=811
  SHAP   F1_dir=0.567 F1_UP_FORT=0.2439 F1_DOWN_FORT=0.6639
  RFE    F1_dir=0.608 F1_UP_FORT=0.1641 F1_DOWN_FORT=0.6456
  LASSO  F1_dir=0.5

In [6]:
# ============================================================
# SYNTHÈSE : SHAP vs RFE vs LASSO, sur la config GLOBAL RF h=5j N=8 SMOTE
# ============================================================
if len(df_fsel):
    by_method = df_fsel.groupby('method')[['F1_dir', 'F1_UP_FORT', 'F1_DOWN_FORT']].mean().round(4)
    by_method = by_method.reindex(CONFIG['methods'])
    print("### Moyenne par méthode (5 folds) ###")
    print(by_method.to_string())
    print(f"\n(pour mémoire, référence README pour SHAP sur la campagne complète: "
          f"F1_dir={BASELINE_F1_DIR})")

    overlap_rows = []
    piv_feat = df_fsel.pivot_table(index='fold', columns='method', values='features', aggfunc='first')
    for k in piv_feat.index:
        sets = {m: set(piv_feat.loc[k, m].split('|')) for m in CONFIG['methods'] if pd.notna(piv_feat.loc[k, m])}
        if 'SHAP' in sets and 'RFE' in sets:
            overlap_rows.append({'fold': k, 'pair': 'SHAP∩RFE',
                                 'overlap': len(sets['SHAP'] & sets['RFE'])})
        if 'SHAP' in sets and 'LASSO' in sets:
            overlap_rows.append({'fold': k, 'pair': 'SHAP∩LASSO',
                                 'overlap': len(sets['SHAP'] & sets['LASSO'])})
    df_overlap = pd.DataFrame(overlap_rows)
    if len(df_overlap):
        print("\n### Chevauchement des features sélectionnées (sur N=8) ###")
        print(df_overlap.groupby('pair')['overlap'].mean().round(1).to_string())

    best = by_method['F1_dir'].idxmax()
    shap_f1 = by_method.loc['SHAP', 'F1_dir'] if 'SHAP' in by_method.index else np.nan
    best_f1 = by_method.loc[best, 'F1_dir']
    print(f"\nMeilleure méthode: {best} (F1_dir={best_f1:.4f})")
    if best == 'SHAP' or (best_f1 - shap_f1) < 0.01:
        print("[VERDICT] SHAP reste la meilleure méthode (ou l'écart avec la meilleure "
              "alternative est dans le bruit, <0.01) — la méthode de sélection n'est pas "
              "un facteur limitant du pipeline, cohérent avec le choix déjà fait.")
    else:
        print(f"[VERDICT] {best} bat SHAP de {best_f1 - shap_f1:+.4f} en F1_dir sur cette "
              "config de référence — piste concrète : envisager de remplacer/compléter "
              "SHAP par cette méthode dans une prochaine itération de VIX_FINAL_ML_SCAN.")

    try:
        with pd.ExcelWriter('VIX_FEATURE_SELECTION_report.xlsx', engine='xlsxwriter') as w:
            df_fsel.to_excel(w, 'Detail', index=False)
            by_method.reset_index().to_excel(w, 'Par_methode', index=False)
            if len(df_overlap):
                df_overlap.to_excel(w, 'Chevauchement', index=False)
        print("\n[SAVE] VIX_FEATURE_SELECTION_report.xlsx")
    except Exception as e:
        print(f"[WARN Export] {e}")
else:
    print("Aucun résultat.")


### Moyenne par méthode (5 folds) ###
        F1_dir  F1_UP_FORT  F1_DOWN_FORT
method                                  
SHAP    0.5972      0.3619        0.6246
RFE     0.5791      0.3248        0.5592
LASSO   0.5716      0.3431        0.6226

(pour mémoire, référence README pour SHAP sur la campagne complète: F1_dir=0.61)

### Chevauchement des features sélectionnées (sur N=8) ###
pair
SHAP∩LASSO    1.4
SHAP∩RFE      1.0

Meilleure méthode: SHAP (F1_dir=0.5972)
[VERDICT] SHAP reste la meilleure méthode (ou l'écart avec la meilleure alternative est dans le bruit, <0.01) — la méthode de sélection n'est pas un facteur limitant du pipeline, cohérent avec le choix déjà fait.

[SAVE] VIX_FEATURE_SELECTION_report.xlsx


In [7]:
# ============================================================
# PUSH DU RAPPORT SUR results/vix-feature-selection
# ============================================================
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

_PUSH_WORKDIR = "/content/_vix_fsel_push"

def push_report():
    files = [f for f in [RESULTS_CSV, 'VIX_FEATURE_SELECTION_report.xlsx'] if os.path.exists(f)]
    if not GITHUB_TOKEN or not files:
        print("[SKIP] Pas de token ou rien à pousser.")
        return
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0:
            print(f"[WARN] clone: {clone.stderr[-300:]}"); return
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        for f in files:
            subprocess.run(["cp", f, f"{_PUSH_WORKDIR}/{f}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email",
                         "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name",
                         "VIX Feature Selection Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add"] + files, check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport Feature Selection — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] {files} sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_report()


[PUSH OK] ['vix_feature_selection_results.csv', 'VIX_FEATURE_SELECTION_report.xlsx'] sur 'results/vix-feature-selection'
